# 貪欲配賦アルゴリズムによる製品ポートフォリオ最適化レポート

**作成日**: 2025年11月29日  
**環境**: analyst_claude_v2 (改善版サンプルデータv2使用)  
**分析対象期間**: 2022-2024年度  
**アルゴリズム**: 貪欲配賦アルゴリズム（Priority Score方式）

---

## エグゼクティブサマリー

本レポートは、改善されたサンプルデータ（v2）に対して貪欲配賦アルゴリズムを適用し、利益を最大化する製品構成を決定した結果を報告します。

###主要な結果

| 指標 | 値 |
|------|-----|
| **総配賦粗利** | **¥2,271,684,422** |
| **総配賦数量** | **82,551本** |
| **平均単位粗利** | **¥27,518/本** |
| **配賦製品組合せ数** | **16組合せ** |
| **需要充足率** | **100%**（全セグメント） |

## 1. 環境セットアップ

必要なライブラリをインポートし、データファイルのパスを設定します。

In [ ]:
# 必要なライブラリのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 日本語フォントの設定（環境に応じて調整）
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

# データディレクトリのパス設定
BASE_DIR = Path('../data')
MASTER_DIR = BASE_DIR / 'master'
RAW_DIR = BASE_DIR / 'raw'
INTERMEDIATE_DIR = BASE_DIR / 'intermediate'

print("環境セットアップ完了")
print(f"ベースディレクトリ: {BASE_DIR.absolute()}")

## 2. データ読み込み

### 2.1 マスタデータの読み込み

In [ ]:
# 製品マスタ
product_master = pd.read_csv(MASTER_DIR / 'product_master.csv')
print("=== 製品マスタ ===")
print(f"製品数: {len(product_master)}")
print(f"\n価格帯別内訳:")
print(product_master['price_band'].value_counts())
product_master.head()

In [ ]:
# セグメントマスタ
segment_master = pd.read_csv(MASTER_DIR / 'segment_master.csv')
print("=== セグメントマスタ ===")
print(f"セグメント数: {len(segment_master)}")
segment_master

### 2.2 配賦結果データの読み込み

In [ ]:
# 配賦結果詳細
allocation_results = pd.read_csv(INTERMEDIATE_DIR / 'allocation_results.csv')
print("=== 配賦結果 ===")
print(f"配賦組合せ数: {len(allocation_results)}")
print(f"総配賦数量: {allocation_results['alloc_qty'].sum():,.0f}本")
print(f"総配賦粗利: ¥{allocation_results['alloc_margin'].sum():,.0f}")
print(f"\n配賦結果詳細:")
allocation_results

In [ ]:
# 拠点別サマリ
plant_summary = pd.read_csv(INTERMEDIATE_DIR / 'allocation_plant_summary.csv')
print("=== 拠点別サマリ ===")
plant_summary

In [ ]:
# セグメント別サマリ
segment_summary = pd.read_csv(INTERMEDIATE_DIR / 'allocation_segment_summary.csv')
print("=== セグメント別サマリ ===")
segment_summary

In [ ]:
# セグメント需要
segment_demand = pd.read_csv(INTERMEDIATE_DIR / 'segment_demand.csv')
print("=== セグメント需要 ===")
print(f"総需要: {segment_demand['demand_qty'].sum():,.0f}本")
segment_demand

## 3. 全体サマリ分析

### 3.1 基本統計量

In [ ]:
# 全体サマリの計算
total_alloc_qty = allocation_results['alloc_qty'].sum()
total_alloc_margin = allocation_results['alloc_margin'].sum()
avg_unit_margin = total_alloc_margin / total_alloc_qty
num_combinations = len(allocation_results)
num_products = allocation_results['product_code'].nunique()

print("="*50)
print("全体サマリ")
print("="*50)
print(f"総配賦数量:     {total_alloc_qty:>15,.0f}本")
print(f"総配賦粗利:     ¥{total_alloc_margin:>14,.0f}")
print(f"平均単位粗利:   ¥{avg_unit_margin:>14,.0f}/本")
print(f"配賦組合せ数:   {num_combinations:>15}組合せ")
print(f"使用製品数:     {num_products:>15}製品")
print("="*50)

### 3.2 キャパシティ使用状況

In [ ]:
# キャパシティ使用状況の詳細表示
print("="*80)
print("キャパシティ使用状況")
print("="*80)

for _, row in plant_summary.iterrows():
    plant = row['plant']
    allocated = row['allocated_qty']
    capacity = row['capacity_limit']
    usage = row['usage_rate']
    remaining = row['remaining_capacity']
    margin = row['allocated_margin']
    
    print(f"\n【拠点{plant}】")
    print(f"  配賦数量:       {allocated:>12,.0f}本")
    print(f"  配賦粗利:       ¥{margin:>11,.0f}")
    print(f"  キャパシティ:   {capacity:>12,.0f}本")
    print(f"  残キャパシティ: {remaining:>12,.0f}本")
    print(f"  使用率:         {usage*100:>12,.1f}%")

total_capacity = plant_summary['capacity_limit'].sum()
total_allocated = plant_summary['allocated_qty'].sum()
overall_usage = total_allocated / total_capacity

print(f"\n【全体】")
print(f"  総配賦数量:     {total_allocated:>12,.0f}本")
print(f"  総キャパシティ: {total_capacity:>12,.0f}本")
print(f"  全体使用率:     {overall_usage*100:>12,.1f}%")
print("="*80)

### 3.3 需要充足状況

In [ ]:
# 需要充足状況の確認
demand_fulfillment = segment_summary[['segment', 'baseline_qty', 'allocated_qty', 'remaining_demand']].copy()
demand_fulfillment['fulfillment_rate'] = (demand_fulfillment['allocated_qty'] / demand_fulfillment['baseline_qty'] * 100)

print("="*80)
print("需要充足状況")
print("="*80)
print(demand_fulfillment.to_string(index=False))
print("="*80)
print(f"\n全セグメント充足率: {demand_fulfillment['fulfillment_rate'].min():.1f}%")
if demand_fulfillment['remaining_demand'].sum() == 0:
    print("✓ 全セグメントの需要が100%充足されています")
else:
    print(f"⚠ 未充足需要: {demand_fulfillment['remaining_demand'].sum():,.0f}本")

## 4. 拠点別分析

### 4.1 拠点別粗利貢献

In [ ]:
# 拠点別集計
plant_analysis = allocation_results.groupby('plant').agg({
    'alloc_qty': 'sum',
    'alloc_margin': 'sum'
}).reset_index()

plant_analysis['margin_share'] = plant_analysis['alloc_margin'] / plant_analysis['alloc_margin'].sum() * 100
plant_analysis['avg_unit_margin'] = plant_analysis['alloc_margin'] / plant_analysis['alloc_qty']

print("=== 拠点別粗利貢献 ===")
print(plant_analysis.to_string(index=False))

# 拠点別粗利シェアの可視化
plt.figure(figsize=(10, 6))
plt.bar(plant_analysis['plant'], plant_analysis['alloc_margin']/1e6, color=['#1f77b4', '#ff7f0e'])
plt.xlabel('Plant', fontsize=12)
plt.ylabel('Allocated Margin (Million Yen)', fontsize=12)
plt.title('Margin Contribution by Plant', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
for i, row in plant_analysis.iterrows():
    plt.text(i, row['alloc_margin']/1e6 + 20, f"¥{row['alloc_margin']/1e6:.0f}M\n({row['margin_share']:.1f}%)", 
             ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 拠点別詳細分析

In [ ]:
# 拠点A詳細
plant_a = allocation_results[allocation_results['plant'] == 'A'].copy()
plant_a = plant_a.sort_values('alloc_margin', ascending=False)
plant_a['margin_share_in_plant'] = plant_a['alloc_margin'] / plant_a['alloc_margin'].sum() * 100

print("\n=== 拠点A詳細分析 ===")
print(f"配賦組合せ数: {len(plant_a)}")
print(f"総配賦数量: {plant_a['alloc_qty'].sum():,.0f}本")
print(f"総配賦粗利: ¥{plant_a['alloc_margin'].sum():,.0f}")
print(f"\nTop5製品:")
print(plant_a[['product_code', 'segment', 'alloc_qty', 'unit_margin', 'alloc_margin', 'margin_share_in_plant']].head().to_string(index=False))

In [ ]:
# 拠点B詳細
plant_b = allocation_results[allocation_results['plant'] == 'B'].copy()
plant_b = plant_b.sort_values('alloc_margin', ascending=False)
plant_b['margin_share_in_plant'] = plant_b['alloc_margin'] / plant_b['alloc_margin'].sum() * 100

print("\n=== 拠点B詳細分析 ===")
print(f"配賦組合せ数: {len(plant_b)}")
print(f"総配賦数量: {plant_b['alloc_qty'].sum():,.0f}本")
print(f"総配賦粗利: ¥{plant_b['alloc_margin'].sum():,.0f}")
print(f"\n全製品:")
print(plant_b[['product_code', 'segment', 'alloc_qty', 'unit_margin', 'alloc_margin', 'margin_share_in_plant']].to_string(index=False))

## 5. セグメント別分析

### 5.1 セグメント別粗利貢献

In [ ]:
# セグメント別分析
segment_analysis = segment_summary.copy()
segment_analysis['margin_share'] = segment_analysis['allocated_margin'] / segment_analysis['allocated_margin'].sum() * 100
segment_analysis['avg_unit_margin'] = segment_analysis['allocated_margin'] / segment_analysis['allocated_qty']
segment_analysis = segment_analysis.sort_values('allocated_margin', ascending=False)

print("=== セグメント別粗利貢献 ===")
print(segment_analysis[['segment', 'allocated_qty', 'allocated_margin', 'margin_share', 'avg_unit_margin', 'share']].to_string(index=False))

# セグメント別粗利シェアの可視化（円グラフ）
plt.figure(figsize=(12, 8))
colors = plt.cm.Set3(range(len(segment_analysis)))
plt.pie(segment_analysis['allocated_margin'], 
        labels=segment_analysis['segment'], 
        autopct='%1.1f%%',
        colors=colors,
        startangle=90)
plt.title('Margin Share by Segment', fontsize=14, fontweight='bold')
plt.axis('equal')
plt.tight_layout()
plt.show()

### 5.2 セグメント別単位粗利ランキング

In [ ]:
# 単位粗利でソート
segment_by_unit_margin = segment_analysis[['segment', 'avg_unit_margin']].sort_values('avg_unit_margin', ascending=False)
segment_by_unit_margin['rank'] = range(1, len(segment_by_unit_margin) + 1)

print("=== セグメント別単位粗利ランキング ===")
print(segment_by_unit_margin[['rank', 'segment', 'avg_unit_margin']].to_string(index=False))

# 可視化
plt.figure(figsize=(12, 6))
plt.barh(segment_by_unit_margin['segment'], segment_by_unit_margin['avg_unit_margin']/1000)
plt.xlabel('Average Unit Margin (Thousand Yen)', fontsize=12)
plt.ylabel('Segment', fontsize=12)
plt.title('Average Unit Margin by Segment', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
for i, row in segment_by_unit_margin.iterrows():
    plt.text(row['avg_unit_margin']/1000 + 0.5, row['segment'], f"¥{row['avg_unit_margin']/1000:.1f}k", 
             va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 6. 製品別分析

### 6.1 配賦製品一覧

In [ ]:
# 製品別集計
product_analysis = allocation_results.groupby('product_code').agg({
    'alloc_qty': 'sum',
    'alloc_margin': 'sum',
    'segment': 'count'  # 組合せ数
}).reset_index()
product_analysis.columns = ['product_code', 'total_qty', 'total_margin', 'num_combinations']
product_analysis = product_analysis.sort_values('total_margin', ascending=False)
product_analysis['margin_share'] = product_analysis['total_margin'] / product_analysis['total_margin'].sum() * 100

# 製品マスタと結合
product_analysis = product_analysis.merge(product_master[['product_code', 'product_name', 'price_band']], on='product_code', how='left')

print("=== 製品別粗利貢献Top10 ===")
print(product_analysis[['product_code', 'product_name', 'price_band', 'num_combinations', 'total_margin', 'margin_share']].head(10).to_string(index=False))

print(f"\n配賦された製品数: {len(product_analysis)} / {len(product_master)} ({len(product_analysis)/len(product_master)*100:.1f}%)")
print(f"  high価格帯: {len(product_analysis[product_analysis['price_band']=='high'])} 製品")
print(f"  low価格帯: {len(product_analysis[product_analysis['price_band']=='low'])} 製品")

### 6.2 最大貢献製品の詳細分析

In [ ]:
# 最大貢献製品（P014）の詳細
top_product = product_analysis.iloc[0]['product_code']
top_product_data = allocation_results[allocation_results['product_code'] == top_product]
top_product_info = product_master[product_master['product_code'] == top_product].iloc[0]

print(f"=== 最大貢献製品: {top_product} ===")
print(f"\n【製品情報】")
print(f"  製品名: {top_product_info['product_name']}")
print(f"  価格帯: {top_product_info['price_band']}")
print(f"  単価範囲: ¥{top_product_info['unit_price_min']:,}-{top_product_info['unit_price_max']:,}")
print(f"  許可拠点: {top_product_info['allowed_plants']}")
print(f"  許可セグメント: {top_product_info['allowed_segments']}")

print(f"\n【配賦実績】")
for _, row in top_product_data.iterrows():
    print(f"  拠点: {row['plant']}")
    print(f"  セグメント: {row['segment']}")
    print(f"  配賦数量: {row['alloc_qty']:,.0f}本")
    print(f"  単位粗利: ¥{row['unit_margin']:,.0f}/本")
    print(f"  粗利率: {row['margin_rate']*100:.2f}%")
    print(f"  配賦粗利: ¥{row['alloc_margin']:,.0f}")
    print(f"  全体シェア: {row['alloc_margin']/total_alloc_margin*100:.1f}%")

## 7. 集中度分析

### 7.1 製品集中度（Pareto分析）

In [ ]:
# Pareto分析（累積寄与率）
product_pareto = product_analysis[['product_code', 'product_name', 'total_margin', 'margin_share']].copy()
product_pareto['cumulative_share'] = product_pareto['margin_share'].cumsum()
product_pareto['rank'] = range(1, len(product_pareto) + 1)

print("=== 製品別Pareto分析 ===")
print(product_pareto[['rank', 'product_code', 'product_name', 'margin_share', 'cumulative_share']].head(10).to_string(index=False))

# Pareto図の作成
fig, ax1 = plt.subplots(figsize=(14, 7))

# 棒グラフ（個別シェア）
ax1.bar(product_pareto['rank'], product_pareto['margin_share'], color='steelblue', alpha=0.7, label='Individual Share')
ax1.set_xlabel('Product Rank', fontsize=12)
ax1.set_ylabel('Margin Share (%)', fontsize=12, color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_ylim(0, 40)

# 折れ線グラフ（累積シェア）
ax2 = ax1.twinx()
ax2.plot(product_pareto['rank'], product_pareto['cumulative_share'], color='red', marker='o', linewidth=2, markersize=6, label='Cumulative Share')
ax2.set_ylabel('Cumulative Share (%)', fontsize=12, color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.set_ylim(0, 105)
ax2.axhline(y=80, color='gray', linestyle='--', alpha=0.7, label='80% Line')

plt.title('Product Concentration - Pareto Chart', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Top3, Top5のシェア
top3_share = product_pareto.iloc[2]['cumulative_share']
top5_share = product_pareto.iloc[4]['cumulative_share']
print(f"\nTop3製品のシェア: {top3_share:.1f}%")
print(f"Top5製品のシェア: {top5_share:.1f}%")

### 7.2 セグメント集中度

In [ ]:
# セグメント集中度分析
segment_pareto = segment_analysis[['segment', 'allocated_margin', 'margin_share']].copy()
segment_pareto['cumulative_share'] = segment_pareto['margin_share'].cumsum()
segment_pareto['rank'] = range(1, len(segment_pareto) + 1)

print("=== セグメント別集中度分析 ===")
print(segment_pareto[['rank', 'segment', 'margin_share', 'cumulative_share']].to_string(index=False))

top3_segments = segment_pareto.iloc[2]['cumulative_share']
print(f"\n上位3セグメントのシェア: {top3_segments:.1f}%")

## 8. リスク分析

### 8.1 集中リスクの定量化

In [ ]:
# 集中リスクの評価
print("=== 集中リスク評価 ===")

# 製品集中リスク
top1_product_share = product_pareto.iloc[0]['margin_share']
top1_product = product_pareto.iloc[0]['product_code']
print(f"\n【製品集中リスク】")
print(f"  最大貢献製品（{top1_product}）シェア: {top1_product_share:.1f}%")
print(f"  リスク額: ¥{total_alloc_margin * top1_product_share / 100:,.0f}")
if top1_product_share > 30:
    print(f"  ⚠️ 警告: 単一製品への依存度が高い（{top1_product_share:.1f}% > 30%）")

# セグメント集中リスク
top1_segment_share = segment_pareto.iloc[0]['margin_share']
top1_segment = segment_pareto.iloc[0]['segment']
print(f"\n【セグメント集中リスク】")
print(f"  最大セグメント（{top1_segment}）シェア: {top1_segment_share:.1f}%")
print(f"  リスク額: ¥{total_alloc_margin * top1_segment_share / 100:,.0f}")
if top1_segment_share > 30:
    print(f"  ⚠️ 警告: 単一セグメントへの依存度が高い（{top1_segment_share:.1f}% > 30%）")

# 拠点集中リスク
max_plant_share = plant_analysis['margin_share'].max()
max_plant = plant_analysis[plant_analysis['margin_share'] == max_plant_share].iloc[0]['plant']
print(f"\n【拠点集中リスク】")
print(f"  最大貢献拠点（{max_plant}）シェア: {max_plant_share:.1f}%")
print(f"  リスク額: ¥{total_alloc_margin * max_plant_share / 100:,.0f}")
if max_plant_share > 60:
    print(f"  ⚠️ 警告: 単一拠点への依存度が高い（{max_plant_share:.1f}% > 60%）")

### 8.2 感度分析シミュレーション

In [ ]:
# P014が失われた場合のシミュレーション
print("=== 感度分析: P014消失シナリオ ===")

p014_margin = allocation_results[allocation_results['product_code'] == 'P014']['alloc_margin'].sum()
remaining_margin = total_alloc_margin - p014_margin
margin_loss_pct = p014_margin / total_alloc_margin * 100

print(f"\n現状総粗利: ¥{total_alloc_margin:,.0f}")
print(f"P014粗利: ¥{p014_margin:,.0f} ({margin_loss_pct:.1f}%)")
print(f"P014消失時の残存粗利: ¥{remaining_margin:,.0f} ({100-margin_loss_pct:.1f}%)")
print(f"\n⚠️ P014の消失により粗利が{margin_loss_pct:.1f}%減少するリスク")

## 9. 主要知見とまとめ

### 9.1 キーファインディング

In [ ]:
print("="*80)
print("主要知見（Key Findings）")
print("="*80)

print("\n【知見1: 高価格帯製品の圧倒的優位性】")
high_combinations = allocation_results.merge(product_master[['product_code', 'price_band']], on='product_code')['price_band'].value_counts()
high_pct = high_combinations.get('high', 0) / len(allocation_results) * 100
print(f"  - 配賦組合せの{high_pct:.1f}%がhigh価格帯製品")
print(f"  - high価格帯製品は単位粗利が高く、priority_scoreで有利")

print("\n【知見2: 製品集中度の高さ】")
print(f"  - Top3製品で総粗利の{top3_share:.1f}%を占める")
print(f"  - 特にP014が単独で{top1_product_share:.1f}%を占める（極めて高い集中度）")

print("\n【知見3: セグメント間の収益性格差】")
max_unit_margin_seg = segment_by_unit_margin.iloc[0]
min_unit_margin_seg = segment_by_unit_margin.iloc[-1]
margin_gap = max_unit_margin_seg['avg_unit_margin'] / min_unit_margin_seg['avg_unit_margin']
print(f"  - 最高単価セグメント（{max_unit_margin_seg['segment']}）: ¥{max_unit_margin_seg['avg_unit_margin']:,.0f}/本")
print(f"  - 最低単価セグメント（{min_unit_margin_seg['segment']}）: ¥{min_unit_margin_seg['avg_unit_margin']:,.0f}/本")
print(f"  - 格差は約{margin_gap:.1f}倍")

print("\n【知見4: 極めて低いキャパシティ使用率】")
print(f"  - 全体使用率: {overall_usage*100:.1f}%（残余{100-overall_usage*100:.1f}%）")
print(f"  - 需要制約が支配的（キャパシティは十分に余っている）")
print(f"  - 需要創出が最優先課題")

print("\n【知見5: 拠点間の不均衡】")
plant_a_share = plant_analysis[plant_analysis['plant']=='A']['margin_share'].values[0]
plant_b_share = plant_analysis[plant_analysis['plant']=='B']['margin_share'].values[0]
print(f"  - 拠点Aシェア: {plant_a_share:.1f}%")
print(f"  - 拠点Bシェア: {plant_b_share:.1f}%")
print(f"  - 拠点Bの製品ラインナップ拡充の余地あり")

print("="*80)

### 9.2 結論

In [ ]:
print("="*80)
print("結論")
print("="*80)

print("\n改善されたサンプルデータv2に対して貪欲配賦アルゴリズムを適用した結果、")
print(f"総粗利¥{total_alloc_margin:,.0f}（約{total_alloc_margin/1e8:.1f}億円）を達成しました。")

print("\n【主要成功要因】")
print("  1. P014（食品向け滅菌バルブ）の圧倒的貢献")
print("  2. high価格帯製品の高単位粗利")
print("  3. electronics/medical/chemicalセグメントの高単価")

print("\n【主要リスク要因】")
print("  1. P014への過度な依存（全粗利の1/3）")
print("  2. 上位3セグメントへの集中（66.5%）")
print("  3. キャパシティの大幅な余剰（91.3%未使用）")

print("\n【次のステップ】")
print("  1. MIPによる最適解算出と貪欲法との比較")
print("  2. P014の品質管理強化とバックアップ体制構築")
print("  3. ポートフォリオ多様化戦略の策定")
print("  4. 需要創出戦略の実行")

print("="*80)

## 10. エクスポート

分析結果をCSVファイルとしてエクスポートします。

In [ ]:
# 分析結果のエクスポート
output_dir = Path('../reports')
output_dir.mkdir(exist_ok=True)

# 製品別分析結果
product_analysis.to_csv(output_dir / 'product_analysis.csv', index=False, encoding='utf-8-sig')
print(f"✓ 製品別分析結果を保存: {output_dir / 'product_analysis.csv'}")

# セグメント別分析結果
segment_analysis.to_csv(output_dir / 'segment_analysis.csv', index=False, encoding='utf-8-sig')
print(f"✓ セグメント別分析結果を保存: {output_dir / 'segment_analysis.csv'}")

# 拠点別分析結果
plant_analysis.to_csv(output_dir / 'plant_analysis.csv', index=False, encoding='utf-8-sig')
print(f"✓ 拠点別分析結果を保存: {output_dir / 'plant_analysis.csv'}")

print("\n分析完了！")

---

**レポート作成日**: 2025年11月29日  
**レポート作成者**: Claude (analyst_claude_v2)  
**バージョン**: 1.0  
**ステータス**: 最終版